# Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch as pt

from torch.utils.data import random_split, DataLoader
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

In [2]:
import sys

sys.path.append(r"/src")

from torchwires import Repo, Runner, PassState

# Dataset & Loaders

In [3]:
class DictDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]

        return {
            "x": x + pt.rand_like(x) * 134,  # Add noise to the input
            "y": y,
        }

In [4]:
transform = transforms.ToTensor()

full_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

train_dataset, val_dataset, test_dataset, _ = random_split(
    full_train_dataset,
    [1000, 300, 300, len(full_train_dataset) - 1600],
)

train_dataset = DictDataset(train_dataset)
val_dataset = DictDataset(val_dataset)
test_dataset = DictDataset(test_dataset)

In [5]:
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=50, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

# Repo Layout

## Definition

In [ ]:
repo = Repo(
    repo_name='mnist-repo_0',
)

## Model

In [7]:
class ClassifierModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(1 * 28 * 28, 10)

    def forward(self, x):
      
        u = self.flatten(x)
        u1 = self.fc1(u)

        return u1

In [8]:
cls_model =  ClassifierModel()

repo.register(
    name='classifier-m',
    value=cls_model
)

Model classifier-m: has been registered
Model classifier-m weights not found: path=mnist-repo_12\exp_1\last\classifier-m.weights.pth


## Optimizer

In [9]:
repo.register(
    name='opt-cls',
    value= torch.optim.Adam(cls_model.parameters(), lr=0.001)
)

Optimizer opt-cls: has been registered
Optimizer opt-cls cache not found: path=mnist-repo_12\exp_1\last\opt-cls.optimizer.pth


In [10]:
repo.optimizers_repo.zero_grad()

# State

In [11]:
class AppState(PassState):
    x = None
    y = None
    y_hat = None
    ce_loss = None
    accuracy = None

## Nodes

In [12]:
def model_pass_node(state: AppState):    
    y_hat = cls_model(state.x)
    
    return {
        'y_hat': y_hat
    }
    

repo.wire(
    name='model-pass',
    body=model_pass_node,
)

Node model-pass: has been registered


In [13]:
def ce_node(state: AppState):
    ce_loss = F.cross_entropy(state.y_hat, state.y)
    
    return {
        'ce_loss': ce_loss
    }
    

repo.wire(
    name='ce_node',
    body=ce_node,
    tracked_features=['ce_loss']
)

Node ce_node: has been registered
Observer tracks: feature=ce_loss


In [14]:
def acc_node(state: AppState):
    accuracy = state.y_hat.argmax(dim=1).eq(state.y).float().mean()
    
    return {
        'accuracy': accuracy
    }
    

repo.wire(
    name='acc_node',
    body=acc_node,
    tracked_features=['accuracy']
)

Node acc_node: has been registered
Observer tracks: feature=accuracy


# Cache Load

In [15]:
repo.load()

Model classifier-m weights not found: path=mnist-repo_12\exp_1\last\classifier-m.weights.pth
Optimizer opt-cls cache not found: path=mnist-repo_12\exp_1\last\opt-cls.optimizer.pth
History cache not found: path=mnist-repo_12\exp_1\history.json


# Training

In [16]:
runner = Runner(
    repo=repo
)

Runner has been initialized: last complete epoch=None


## cb

In [17]:
# runner.enable_autosave(
#     interval=2,
# )

In [18]:
runner.enable_checkpointing(
    split='val',
    monitor='ce_loss',
    aggregate_mode='mean',
    mode='min'
)

In [19]:
# runner.enable_early_stopping(
#     split= 'val',
#     monitor='accuracy',
#     patience=3,
#     aggregate_mode='mean',
#     mode='max'
# )

## train

In [20]:
runner.train(
    epochs=100,
    backprob_losses=['ce_loss'],
    device='cpu',
    train_loader=train_loader,
    val_loader=val_loader
)

Start training: from epoch=0 to epoch=100

epoch:0 | batch:19 | split:train | device:cpu | ce_loss:37.85124206542969 | accuracy:0.14000000059604645 
epoch:0 | batch:5 | split:val | device:cpu | ce_loss:47.03105163574219 | accuracy:0.10000000149011612    
Checkpoint: val-ce_loss improved from inf to 44.411617279052734 | checkpoint name: checkpoint_val-ce_loss
Model classifier-m weights saved: path=mnist-repo_12\exp_1\checkpoint_val-ce_loss\classifier-m.weights.pth
Optimizer opt-cls weights saved: path=mnist-repo_12\exp_1\checkpoint_val-ce_loss\opt-cls.optimizer.pth
History saved: json=mnist-repo_12\exp_1\history.json

epoch:1 | batch:19 | split:train | device:cpu | ce_loss:37.05886459350586 | accuracy:0.11999999731779099 
epoch:1 | batch:5 | split:val | device:cpu | ce_loss:30.91222381591797 | accuracy:0.10000000149011612    
Checkpoint: val-ce_loss improved from 44.411617279052734 to 39.640933990478516 | checkpoint name: checkpoint_val-ce_loss
Model classifier-m weights saved: path=mni

# Save

In [21]:
repo.save()

Model classifier-m weights saved: path=mnist-repo_12\exp_1\last\classifier-m.weights.pth
Optimizer opt-cls weights saved: path=mnist-repo_12\exp_1\last\opt-cls.optimizer.pth
History saved: json=mnist-repo_12\exp_1\history.json


# Visuals

In [22]:
# tw.Visualizer().visualize_repo(
#     repo=repo
# )

In [23]:
# tw.Visualizer().display_df(
#     repo=repo
# )

# Predict

In [24]:
x, y = test_dataset[0]['x'], test_dataset[0]['y']

x = pt.stack([x])
y = pt.tensor([y])

s = runner.inference(
    device='cpu',
    inputs={
        'x': x,
        'y': y
    }
)

# print(s)